Import Librararies

In [ ]:
from typing import TypedDict, Annotated, List
from operator import add
from langchain_anthropic import ChatAnthropic
from langgraph.graph import StateGraph, END
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage, AIMessage
from langchain_tavily import TavilySearch
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import os

load_dotenv()
print("Imports are successfull!!!")

Initialize Claude and Tavily

In [ ]:
llm = ChatAnthropic(
    model = "claude-sonnet-4-6",
    temperature = 0
)

search_tool = TavilySearch(
    max_results=3,
    topic="general"
)

print("Claude and Tavily ready!!!")

Structured Output Classes

In [ ]:
class SearchQuery(BaseModel):
    query: str = Field(description="A specific search query to verify security recommendations or find current CVEs and best practices")

class SecurityAssessment(BaseModel):
    policy: str = Field(description="Main response to the question")
    reflection: str = Field(description="Self-critique of the answer")
    search_queries: List[SearchQuery] = Field(description="Queries for additional research")

print("Structured output classes defined!!!")

Define AgentState

In [ ]:
class AgentState(TypedDict):
    infrastructure: str
    history: List[str]
    search_results: List[str]
    iterations: int
    assessment_policy: str
    assessment_reflection: str
    assessment_queries: List[str]

print("Agent State Defined!!!")

Generator Node

In [ ]:
def generate_policy(state: AgentState) -> dict:
    iterations = state.get("iterations", 0) + 1
    print(f"\n🔄 Iteration {iterations}...")

    prompt = state["infrastructure"]
    history = state.get("history", [])
    search_results = state.get("search_results", [])

    history_context = ""
    if history:
        history_context = "\n\nPrevious attempts and critiques:\n"
        for i, h in enumerate(history):
            history_context += f"\nAttempt {i+1}: {h}\n"

    search_context = ""
    if search_results:
        search_context = "\n\nVerified information from security research:\n"
        for result in search_results:
            search_context += f"- {result}\n"

    user_content = f"""
Infrastructure Description:
{prompt}
{history_context}
{search_context}
Generate a comprehensive security policy addressing all findings above.
"""

    structured_llm = llm.with_structured_output(SecurityAssessment)
    response = structured_llm.invoke([
        SystemMessage(content="""
You are a senior cloud security architect with 20+ years of experience
across Azure, AWS and GCP.
Generate a structured security policy covering:
1. Identity & Access Management
2. Network Security & Segmentation
3. Data Protection & Classification
4. Secret & Credential Management
5. Logging, Monitoring & Alerting
6. Patch & Vulnerability Management
7. Incident Response
8. Compliance Considerations
Also provide self-critique and 3 specific search queries to verify
your recommendations against current CVEs and best practices.
Maximum 400 words for the policy.
        """),
        HumanMessage(content=user_content)
    ])

    print(f"✅ Iteration {iterations} complete")
    print(f"   Search queries generated: {len(response.search_queries)}")

    history.append(f"Policy: {response.policy[:200]}... Reflection: {response.reflection[:100]}")

    return {
        "history": history,
        "search_results": state.get("search_results", []),
        "iterations": iterations,
        "infrastructure": prompt,
        "assessment_policy": response.policy,
        "assessment_reflection": response.reflection,
        "assessment_queries": [q.query for q in response.search_queries]
    }

print("Policy Generator node defined!")

Search Node

In [ ]:
def search_for_evidence(state: AgentState) -> dict:
    print(f"\n🔍 Searching for evidence...")

    queries = state.get("assessment_queries", [])
    search_results = state.get("search_results", [])

    for query in queries:
        print(f"   Searching: {query}")
        try:
            results = search_tool.invoke(query)
            print(f"   Result type: {type(results)}")
            
            # Handle different return types from Tavily
            if isinstance(results, list):
                for result in results:
                    if isinstance(result, dict):
                        search_results.append(result.get("content", str(result)))
                    else:
                        search_results.append(str(result))
            elif isinstance(results, str):
                search_results.append(results)
            else:
                search_results.append(str(results))
                
        except Exception as e:
            print(f"   Search error: {str(e)}")
            continue

    print(f"✅ Found {len(search_results)} total evidence items")
    return {
        "search_results": search_results,
        "infrastructure": state["infrastructure"],
        "history": state.get("history", []),
        "iterations": state.get("iterations", 0),
        "assessment_queries": [],
        "assessment_policy": state.get("assessment_policy", ""),
        "assessment_reflection": state.get("assessment_reflection", "")
    }

print("Search node defined!")

Conditional Edge

In [ ]:
def should_continue(state: AgentState) -> str:
    iterations = state.get("iterations", 0)
    queries = state.get("assessment_queries", [])
    
    print(f"\n⚡ Checking condition - iterations: {iterations}")
    print(f"   Raw queries value: {queries}")
    print(f"   Queries type: {type(queries)}")
    print(f"   Queries length: {len(queries) if queries else 0}")

    # Fix - check explicitly for non-empty list
    has_queries = isinstance(queries, list) and len(queries) > 0

    if iterations >= 3 or not has_queries:
        print("→ Ending")
        return "end"

    print("→ Searching for evidence")
    return "search"

print("Condition defined!")

Build the Graph

In [ ]:
workflow = StateGraph(AgentState)
workflow.add_node("generate_policy", generate_policy)
workflow.add_node("search_for_evidence", search_for_evidence)
workflow.add_edge("search_for_evidence", "generate_policy")
workflow.add_conditional_edges("generate_policy",should_continue, {
    "search" : "search_for_evidence",
    "end": END,
})
workflow.set_entry_point("generate_policy")
print("Graph is Built & Ready!!!")

Compile and invoke

In [ ]:
app = workflow.compile()
result = app.invoke({
    "infrastructure": "Azure Data Factory instance to be provisioned for all DEV users",
    "history": [],
    "search_results": [],
    "iterations": 0,
    "assessment_policy": "",
    "assessment_reflection": "",
    "assessment_queries": []
})
print(f"\nDebug - assessment_queries in result: {result.get('assessment_queries', [])}")

print(f"\n✅ Completed in {result['iterations']} iterations")
print(f"📚 Evidence gathered: {len(result['search_results'])} items")
print("\n📋 Final Security Policy:")
print("=" * 60)
print(result["assessment_policy"])
print("\n🔍 Final Reflection:")
print("=" * 60)
print(result["assessment_reflection"])